## EMOBON Questions

This file was made to answer specific questions about emobon to see its capabilities for data harvesting and data management.
The questions posed originated from https://vliz.atlassian.net/wiki/spaces/VMDCOS/pages/204374034/First+set+of+queries+to+run+on+3store+of+emo+bon+data

The top level questions are as follows:

- 1: Give me the list of observatories: their names, country, location (lat, long and mrgid), and any habitat info
- 2: For each observatory now, give me the names, the number of sampling events for water and sediment and the overall date coverage, and the number of samples taken
- 3: For a named observatory (so I input the name), give me the number and percentage of missing information in the mandatory fields, per event (not per sample!). NA is “missing information, by the way so don’t throw them out. Plot or tabulate this per mandatory field name (as given in the logsheets)
- 4: And also the summary of the %coverage in the mandatory fields for the sampling tab (that is, the sampling tab of the googlesheet) and of the measured tab separately. do not throw out the NAs here.
- 5: For the mandatory measurements, give me the mean and mean deviation, as well as max and min values, for a named observatory (so I input the name), summed over all sampling events and over all water events separately (plots could be side-by-side). Give me this following the BODC names but then also output a list of BODC vs googlesheet names so I can read that off. Throw out the NAs here
- 6: Over all the observatories, list the samp_collect_devices used and the number of events (not samples) each was used for (I want to see how many different ones there are). Here you can throw out the NAs before reporting.
--> matched to query e (collect_device_usage)
- 7: For each observatory, how many samples for water and for sediment separately have been shipped to HQ
- 8: Identify the metagenomic sampling events taken in the English Channel (aka La Manche) during 2022-2023 where the sea temperature was below 10 degrees Celsius and the abundance of the taxon E. coli was > 2% (all of this is within the EMO BON data sets)

For each of these questions also make a query template that can be used for an anduser to have some variables that can be used in UDAL.

In [30]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [31]:
# import all stuff needed
from conneg_functions import execute_to_df, generate_sparql

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from IPython.core.display import HTML
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output
from sema.query import DefaultSparqlBuilder, GraphSource as KGSource, QueryResult
from pathlib import Path
import os
import re
from pandas import DataFrame

In [32]:
# some paramter setup for the triplestore
# SPARQL EndPoint to use - wrapped as Knowledge-Graph 'source'
GDB_BASE: str = os.getenv("GDB_BASE", "http://graphdb:7200/")
# print(f"{os.getenv('GDB_BASE')=}")
# print(f"{GDB_BASE=}")
GDB_REPO: str = os.getenv("GDB_REPO", "kgap")
GDB_ENDPOINT: str = f"{GDB_BASE}repositories/{GDB_REPO}"
# print(f"{GDB_ENDPOINT=}")
GDB: KGSource = KGSource.build(GDB_ENDPOINT)

# some css to format tables so that they are readable
HTML("""
<style>
    .dataframe td, .dataframe th {
        min-width: 500px;
        word-wrap: break-word;
    }
</style>
""")

In [33]:
# - 1: Give me the list of observatories: their names, country, location (lat, long and mrgid), and any habitat info

sparql_observatory = '''
PREFIX owl: <http://www.w3.org/2002/07/owl#> 
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX emobon: <https://data.emobon.embrc.eu/ns/core#>
SELECT DISTINCT ?observatory 
(GROUP_CONCAT(DISTINCT ?marine_region; SEPARATOR="|") AS ?marine_regions)
(GROUP_CONCAT(DISTINCT ?marine_region_id; SEPARATOR="|") AS ?marine_region_ids)
(GROUP_CONCAT(DISTINCT ?country; SEPARATOR="|") AS ?countries)
(GROUP_CONCAT(DISTINCT ?biome; SEPARATOR="|") AS ?biomes)
WHERE  {
    ?observatory a emobon:Observatory .
    OPTIONAL { ?observatory emobon:originCountry ?country . }
    OPTIONAL { ?observatory emobon:marineRegionName ?marine_region . }
    OPTIONAL { ?observatory emobon:marineRegion ?marine_region_id . }
    OPTIONAL { ?observatory emobon:broadBiome ?biome . }
}
GROUP BY ?observatory
'''

result: QueryResult = GDB.query(sparql=sparql_observatory)
print(f"Result: {result=}")
df_observatory: DataFrame = result.to_dataframe()

HTML("""
<style>
    .dataframe td, .dataframe th {
        word-wrap: break-word;
    }
</style>
""" + df_observatory.to_html())

df_observatory


Result: result=<sema.query.query.SPARQLQueryResult object at 0x7ff1b0637ed0>


,observatory,marine_regions,marine_region_ids,countries,biomes
0,http://data.emobon.embrc.eu/observatory-hcmr-1...,Crete Sea|Mediterranean Sea - Eastern Basin|Ae...,http://marineregions.org/mrgid/3315|http://mar...,Greece,marine%20biome%20%5BENVO:00000447%5D|marine%20...
1,http://data.emobon.embrc.eu/observatory-mbal4-...,Western Channel|English Channel|North Atlantic...,http://marineregions.org/mrgid/17527|http://ma...,United Kingdom,marine%20biome%20%5BENVO:00000447%5D|marine%20...
2,http://data.emobon.embrc.eu/observatory-iuieil...,Indian Ocean|Gulf of Eilat,http://marineregions.org/mrgid/4263|http://mar...,Israel,marine%20biome%20%5BENVO:00000447%5D|marine%20...
3,http://data.emobon.embrc.eu/observatory-bpns-c...,North Atlantic Ocean|North Sea|Belgian part of...,http://marineregions.org/mrgid/1912|http://mar...,Belgium,marine%20biome%20%5BENVO:00000447%5D|marine%20...
4,http://data.emobon.embrc.eu/observatory-bpns-c...,North Atlantic Ocean|North Sea|Belgian part of...,http://marineregions.org/mrgid/1912|http://mar...,Belgium,marine%20biome%20%5BENVO:00000447%5D|marine%20...
5,http://data.emobon.embrc.eu/observatory-esc68n...,Norwegian Sea|Arctic Ocean|Norwegian part of t...,http://marineregions.org/mrgid/1906|http://mar...,Norway,marine%20biome%20%5BENVO:00000447%5D|marine%20...
6,http://data.emobon.embrc.eu/observatory-roskog...,English Channel|North Atlantic Ocean|French pa...,http://marineregions.org/mrgid/2389|http://mar...,France,marine%20biome%20%5BENVO:00000447%5D|marine%20...
7,http://data.emobon.embrc.eu/observatory-roskog...,English Channel|North Atlantic Ocean|French pa...,http://marineregions.org/mrgid/2389|http://mar...,France,marine%20biome%20%5BENVO:00000447%5D|marine%20...
8,http://data.emobon.embrc.eu/observatory-rformo...,North Atlantic Ocean|Ria Formosa|Atlantic Ocean,http://marineregions.org/mrgid/1912|http://mar...,Portugal,marine%20biome%20%5BENVO:00000447%5D
9,http://data.emobon.embrc.eu/observatory-rformo...,North Atlantic Ocean|Ria Formosa|Atlantic Ocean,http://marineregions.org/mrgid/1912|http://mar...,Portugal,marine%20biome%20%5BENVO:00000447%5D|marine%20...


In [ ]:
# - 2: For each observatory now, give me the names, the number of sampling events for water and sediment and the overall date coverage, and the number of samples taken

# obsevation_url can be : http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/measured/EMOBON_NRMCB_Wa_230803_20um_4#phaeopigments
# event will be : http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_230803

#for each observatory , the value of the sparql should be contructed as follows:
# take the base from the observation_url :
# eg: http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/measured/EMOBON_NRMCB_Wa_230803_20um_4#phaeopigments
# will give you the base : http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/
# for the sampling-event add the following to the base:
# sampling-event/{{event-id}}
# with event-id = EMOBON_NRMCB_Wa_230803 -> which is extracted from the observation_url 
#by taking the part after /EMOBON_ and before the second to last _

# this however cannot be done in the sparql query since it does not support string manpulation of found values 

sparql_sampling_events = '''
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX emobon: <https://data.emobon.embrc.eu/ns/core#>
PREFIX emobon_s: <https://data.emobon.embrc.eu/ns/sampling#>
SELECT DISTINCT ?observation
WHERE {
    ?observation a sosa:Observation .
    }
'''
result: QueryResult = GDB.query(sparql=sparql_sampling_events)
df_sampling_events: DataFrame = result.to_dataframe()

observations = df_sampling_events["observation"].tolist()


def construct_sampling_event_url(observation_url):
    base_url = observation_url.split("/measured/")[0] + "/"
    if "observation" in observation_url:
        base_url = observation_url.split("/observation/")[0] + "/"
    match = re.search(r"EMOBON_(.+?)_(.+?)_(.+?)(?:_|#)", observation_url)
    if match:
        event_id = match.group(1)
        sediment_sample_type = match.group(2)
        date = match.group(3)
        
        sampling_event_url = f"{base_url}sampling-event/{event_id}_{sediment_sample_type}_{date}"
        return sampling_event_url
    return None

# Process each observation URL
sampling_event_urls = [construct_sampling_event_url(url) for url in observations]
combined_df = pd.DataFrame({"observation": observations, "sampling_event_url": sampling_event_urls})
observations_with_no_sampling_event = combined_df[combined_df["sampling_event_url"].isnull()]["observation"].tolist()

###
# when ran on 28-04-2024 some observations from nrmcb were not parsed in the triplestore , this might have been due to the fact that the etl ppeline was ran before the latest changes to the template files
###

# Count the occurrences of each unique sampling_event_url
sampling_event_counts = combined_df["sampling_event_url"].value_counts()
# Sort the counts in descending order
sorted_sampling_event_counts = sampling_event_counts.sort_values(ascending=False)
sparql_sampling_events_linked_to_observatories = '''
select * where { 
	?sampling_event_url <https://data.emobon.embrc.eu/ns/sampling#linkedToObservatory> ?observatory .
}
'''
result: QueryResult = GDB.query(sparql=sparql_sampling_events_linked_to_observatories)
df_sampling_events_linked_to_observatories: DataFrame = result.to_dataframe()
# Merge the sampling_event_counts with df_sampling_events_linked_to_observatories
df_sampling_event_counts = sampling_event_counts.reset_index()
df_sampling_event_counts.columns = ['sampling_event_url', 'count']

# Ensure the column names match for merging
df_sampling_events_linked_to_observatories.columns = ['sampling_event_url', 'observatory']

# Perform the merge
combined_sampling_events_df = pd.merge(
    df_sampling_event_counts,
    df_sampling_events_linked_to_observatories,
    on='sampling_event_url',
    how='left'
)

# Display the combined dataframe
combined_sampling_events_df

# Filter sampling events that don't have an observatory linked
unlinked_sampling_events = combined_sampling_events_df[combined_sampling_events_df["observatory"].isnull()]

# Display the sampling event URLs without linked observatories
unlinked_sampling_events_urls = unlinked_sampling_events["sampling_event_url"].tolist()
print(len(unlinked_sampling_events_urls), "sampling events without linked observatories")
print("Sampling event URLs without linked observatories:")
for url in unlinked_sampling_events_urls:
    print(url)
    
# Some weird stuff is going on here with the linking of the sampling events to the observatories
# for instance when testing this out on bpns with the following query:
#PREFIX sosa: <http://www.w3.org/ns/sosa/>
#SELECT DISTINCT ?observation
#WHERE {
#    ?observation a sosa:Observation .
#    FILTER regex(str(?observation), "^http://data\\.emobon\\.embrc\\.eu/observatory-bpns-crate/water/.*")
#}
# there are 3,3K results but not all the observations link up to sampling events. There is a gap here.


649 sampling events without linked observatories
Sampling event URLs without linked observatories:
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_220427
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_220803
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_230803
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_220302
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_230214
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_220623
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_221011
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_230427
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_221206
http://data.emobon.embrc.eu/observatory-nrmcb-crate/water/sampling-event/NRMCB_Wa_231017
http://data